# 第20章 文本、日期与特征处理

结合字符串、日期和数值列构造可分析的业务特征。


## 先解决一个小问题

拿一组小型业务数据练习“文本、日期与特征处理”：先看数据结构，再完成一次明确的计算或转换。结合字符串、日期和数值列构造可分析的业务特征。


## 这章为什么先学

这是“Pandas”路线中第 20 章的操作重点。本章只解决“文本、日期与特征处理”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：批量清洗文本列


## 做完要留下什么

产出一个与“文本、日期与特征处理”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 批量清洗文本列
- 解析和拆解日期
- 计算时间差
- 构造分类与数值特征


## 核心概念

- Pandas字符串方法通过str访问器调用。
- 日期必须先转换为datetime才能进行时间运算。
- 特征应由业务问题驱动，并避免使用未来信息。


## 示例 1：文本标准化

清洗步骤包括去空格、统一大小写和提取模式。


In [ ]:
import pandas as pd

customers = pd.DataFrame({
    "name": [" 张三 ", "LI SI", "王 五"],
    "email": ["A@EXAMPLE.COM", "li@test.cn", "wang@example.com"],
})
customers["name_clean"] = customers["name"].str.strip().str.replace(" ", "", regex=False)
customers["email_clean"] = customers["email"].str.strip().str.lower()
customers["domain"] = customers["email_clean"].str.extract(r"@(.+)$", expand=False)
print(customers)


## 示例 2：日期解析与拆解

errors='coerce'把无效日期转换为NaT。


In [ ]:
orders = pd.DataFrame({
    "order_date": ["2026-01-05", "2026-02-18", "invalid", "2026-03-22"],
    "amount": [320, 880, 460, 1250],
})
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["month"] = orders["order_date"].dt.to_period("M").astype("string")
orders["weekday"] = orders["order_date"].dt.day_name()
print(orders)


## 示例 3：特征构造

特征应有明确口径并可从原始字段复算。


In [ ]:
reference_date = pd.Timestamp("2026-04-01")
orders["days_ago"] = (reference_date - orders["order_date"]).dt.days
orders["amount_level"] = pd.cut(
    orders["amount"],
    bins=[0, 500, 1000, float("inf")],
    labels=["普通", "重点", "大额"],
)
print(orders)


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd
from js import window

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = f"{window.location.origin}/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
features = large_orders.assign(
    order_date=large_orders["order_time"].dt.date,
    month=large_orders["order_time"].dt.to_period("M").astype("string"),
    weekday=large_orders["order_time"].dt.day_name(),
    hour=large_orders["order_time"].dt.hour,
    is_weekend=large_orders["order_time"].dt.dayofweek >= 5,
    description_clean=large_orders["description"].str.strip().str.title(),
    order_label=large_orders["country"].astype("string").str.cat(large_orders["stock_code"], sep=" / "),
)
display(features[["order_time", "month", "weekday", "hour", "is_weekend", "description_clean", "order_label"]].head())
print("月份跨度：", features["month"].min(), "至", features["month"].max())


## 常见误区

- 直接对object列使用日期运算
- 正则提取失败后不检查缺失
- 使用结果变量构造导致数据泄漏的特征


## 综合练习

1. 清洗手机号中的空格和连字符
2. 解析注册日期
3. 构造注册月份和账户天数

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“清洗手机号中的空格和连字符”。
2. **独立完成**：不复制示例代码，完成“解析注册日期”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“构造注册月份和账户天数”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

users = pd.DataFrame({
    "phone": ["138-0000-1234", " 139 0000 5678 "],
    "registered_at": ["2025-12-15", "2026-02-08"],
})

# TODO: 清洗手机号，去除所有非数字字符
users["phone_clean"] =

# TODO: 解析注册日期
users["registered_at"] =

# TODO: 提取注册月份
users["register_month"] =

# TODO: 计算账户天数（以2026-04-01为参考日期）
users["account_days"] =

print(users)


In [ ]:
import pandas as pd

users = pd.DataFrame({
    "phone": ["138-0000-1234", " 139 0000 5678 "],
    "registered_at": ["2025-12-15", "2026-02-08"],
})
users["phone_clean"] = users["phone"].str.replace(r"\D", "", regex=True)
users["registered_at"] = pd.to_datetime(users["registered_at"])
users["register_month"] = users["registered_at"].dt.to_period("M").astype("string")
users["account_days"] = (pd.Timestamp("2026-04-01") - users["registered_at"]).dt.days
print(users)

# 自检
assert users["phone_clean"].tolist() == ["13800001234", "13900005678"], "检查手机号清洗"
assert users["account_days"].tolist() == [107, 52], "检查账户天数计算"


## 本章小结

结合字符串、日期和数值列构造可分析的业务特征。

**迁移思考**：

1. 如果需要提取用户邮箱的用户名部分（@符号之前），正则表达式应该如何写？
2. 为什么特征构造要避免使用未来信息？举一个会导致数据泄漏的例子。


### 你已经掌握

- 批量清洗文本列
- 解析和拆解日期
- 计算时间差
- 构造分类与数值特征


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 文本标准化 | 清洗步骤包括去空格、统一大小写和提取模式。 | `pd.DataFrame()`、`str.strip()`、`str.replace()`、`str.lower()` |
| 日期解析与拆解 | errors='coerce'把无效日期转换为NaT。 | `pd.DataFrame()`、`pd.to_datetime()`、`dt.to_period()`、`dt.day_name()` |
| 特征构造 | 特征应有明确口径并可从原始字段复算。 | `pd.Timestamp()`、`pd.cut()`、`orders["days_ago"]`、`orders["order_date"]` |


### 需要注意

- 直接对object列使用日期运算
- 正则提取失败后不检查缺失
- 使用结果变量构造导致数据泄漏的特征


### 完成检查

- [ ] 能够批量清洗文本列
- [ ] 能够解析和拆解日期
- [ ] 能够计算时间差
- [ ] 能够构造分类与数值特征


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
